In [1]:
import pandas as pd
import hashlib
import json
from pathlib import Path
from datetime import datetime

In [2]:
# import shutil
# from pathlib import Path
# from datetime import datetime

# # Paths
# HISTORY_PATH = Path("selection_history.json")
# TRAIN_DIR    = Path("training_sets")
# OUTPUT_DIR   = Path(".")

# # Backup folder with timestamp
# BACKUP_DIR = Path("backup_before_reset") / datetime.now().strftime("%Y%m%d_%H%M%S")
# BACKUP_DIR.mkdir(parents=True, exist_ok=True)

# # Move history file if present
# if HISTORY_PATH.exists():
#     shutil.move(str(HISTORY_PATH), BACKUP_DIR / HISTORY_PATH.name)

# # Move previous training sets
# if TRAIN_DIR.exists():
#     shutil.move(str(TRAIN_DIR), BACKUP_DIR / TRAIN_DIR.name)

# # Move any unique_sample_*.csv files
# moved_any = False
# for p in OUTPUT_DIR.glob("unique_sample_*.csv"):
#     shutil.move(str(p), BACKUP_DIR / p.name)
#     moved_any = True

# print("✅ Reset complete. Archived prior state to:", BACKUP_DIR.resolve())

import shutil
from pathlib import Path
from datetime import datetime

# Paths (fixed leading slashes)
HISTORY_PATH = Path("/home/ubuntu/TW_MultiLabel_SMP/jupyter-notebooks/selection_history.json")
TRAIN_DIR    = Path("/home/ubuntu/TW_MultiLabel_SMP/datasets/training_sets")
OUTPUT_DIR   = Path(".")

# Backup folder with timestamp
BACKUP_DIR = Path("/home/ubuntu/TW_MultiLabel_SMP/datasets/backup_before_reset") / datetime.now().strftime("%Y%m%d_%H%M%S")
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

# ---- Preserve history (copy, don't move) ----
if HISTORY_PATH.exists():
    backup_history = BACKUP_DIR / HISTORY_PATH.name
    try:
        shutil.copy2(HISTORY_PATH, backup_history)
        print(f"📝 Preserved history: copied to {backup_history}")
    except Exception as e:
        print(f"⚠️ Could not copy history file: {e}")
else:
    print("ℹ️ No selection_history.json found to preserve.")

# ---- Move previous training sets to backup ----
if TRAIN_DIR.exists():
    dest = BACKUP_DIR / TRAIN_DIR.name
    try:
        shutil.move(str(TRAIN_DIR), dest)
        print(f"📦 Archived training_sets to: {dest}")
    except Exception as e:
        print(f"⚠️ Could not move training_sets: {e}")
else:
    print("ℹ️ No training_sets directory found to archive.")

# ---- Move any unique_sample_*.csv files to backup ----
moved_any = False
for p in OUTPUT_DIR.glob("unique_sample_*.csv"):
    try:
        shutil.move(str(p), BACKUP_DIR / p.name)
        print(f"📄 Archived {p.name}")
        moved_any = True
    except Exception as e:
        print(f"⚠️ Could not move {p}: {e}")

if not moved_any:
    print("ℹ️ No unique_sample_*.csv files found to archive.")

print("✅ Reset complete. Old artifacts archived. History preserved in place.")



📝 Preserved history: copied to /home/ubuntu/TW_MultiLabel_SMP/datasets/backup_before_reset/20251022_021902/selection_history.json
ℹ️ No training_sets directory found to archive.
📄 Archived unique_sample_20251020_214455.csv
✅ Reset complete. Old artifacts archived. History preserved in place.


In [3]:
def row_hash(row: pd.Series) -> str:
    """Generate a deterministic hash of a row if no explicit ID column exists."""
    obj = row.to_dict()
    normalized = {str(k): ("" if pd.isna(v) else str(v)) for k, v in obj.items()}
    payload = json.dumps(normalized, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def load_df(path: str, dataset_name: str, id_column: str = None) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df["__dataset"] = dataset_name
    df["__source_file"] = Path(path).name

    if id_column and id_column in df.columns:
        df["unique_key"] = df[id_column].astype(str)
    else:
        df["unique_key"] = df.apply(row_hash, axis=1)

    return df


In [4]:
# Update these paths to your local copies
ABORTION_PATH = "/home/ubuntu/TW_MultiLabel_SMP/datasets/abortion_data-updated - new_abortion_related_subreddits_text_posts .csv"
MISCARRIAGE_PATH = "/home/ubuntu/TW_MultiLabel_SMP/datasets/miscarriage_related_posts.csv"
HARASSMENT_PATH = "/home/ubuntu/TW_MultiLabel_SMP/datasets/sexual-harrassment-data-updated - RelevantByTitle.csv"

# History file (persists across runs)
HISTORY_PATH = Path("/home/ubuntu/TW_MultiLabel_SMP/jupyter-notebooks/selection_history.json")

# Desired counts per dataset
counts = {
    "abortion": 167,
    "miscarriage": 166,
    "harassment": 167
}

# Optional: if your CSVs have a post_id or id column
ID_COLUMN = None   # e.g. "post_id"


In [5]:
# Load CSVs
abortion_df = load_df(ABORTION_PATH, "abortion", ID_COLUMN)
miscarriage_df = load_df(MISCARRIAGE_PATH, "miscarriage", ID_COLUMN)
harassment_df = load_df(HARASSMENT_PATH, "harassment", ID_COLUMN)

# Load or initialize selection history
if HISTORY_PATH.exists():
    with open(HISTORY_PATH, "r", encoding="utf-8") as f:
        history = json.load(f)
else:
    history = {"used_keys": [], "runs": []}

used_keys = set(history.get("used_keys", []))

# Exclude previously used posts
def exclude_used(df):
    return df[~df["unique_key"].isin(used_keys)].copy()

ab_pool = exclude_used(abortion_df)
mi_pool = exclude_used(miscarriage_df)
sh_pool = exclude_used(harassment_df)

In [6]:
shortages = []
if len(ab_pool) < counts["abortion"]:
    shortages.append(f"abortion (need {counts['abortion']}, have {len(ab_pool)})")
if len(mi_pool) < counts["miscarriage"]:
    shortages.append(f"miscarriage (need {counts['miscarriage']}, have {len(mi_pool)})")
if len(sh_pool) < counts["harassment"]:
    shortages.append(f"harassment (need {counts['harassment']}, have {len(sh_pool)})")

if shortages:
    raise RuntimeError("Not enough fresh rows: " + "; ".join(shortages))

sample_ab = ab_pool.sample(n=counts["abortion"], replace=False, random_state=None)
sample_mi = mi_pool.sample(n=counts["miscarriage"], replace=False, random_state=None)
sample_sh = sh_pool.sample(n=counts["harassment"], replace=False, random_state=None)

sample_all = pd.concat([sample_ab, sample_mi, sample_sh], ignore_index=True)
sample_all = sample_all.sample(frac=1.0).reset_index(drop=True)  # shuffle


In [7]:
# Save timestamped CSV
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = Path(f"unique_sample_{ts}.csv")
sample_all.to_csv(out_path, index=False)

# Update history
new_keys = sample_all["unique_key"].tolist()
history["used_keys"].extend(new_keys)
history["runs"].append({
    "timestamp": datetime.utcnow().isoformat() + "Z",
    "output_file": str(out_path),
    "counts": counts,
    "selected": len(new_keys)
})

with open(HISTORY_PATH, "w", encoding="utf-8") as f:
    json.dump(history, f, ensure_ascii=False, indent=2)

print(f"✅ Saved {len(sample_all)} posts to {out_path}")
print(f"Remaining after this run:")
print("  abortion:", len(ab_pool) - counts["abortion"])
print("  miscarriage:", len(mi_pool) - counts["miscarriage"])
print("  harassment:", len(sh_pool) - counts["harassment"])


✅ Saved 500 posts to unique_sample_20251022_021937.csv
Remaining after this run:
  abortion: 3069
  miscarriage: 386
  harassment: 3820


In [8]:
# Make a clean training label column (good for ML pipelines)
sample_all = sample_all.copy()
sample_all["label"] = sample_all["__dataset"]  # keep your original columns intact

# Create a training_sets folder
TRAIN_DIR = Path("training_sets")
TRAIN_DIR.mkdir(parents=True, exist_ok=True)

# Save a per-run training file (500 rows)
train_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
train_csv = TRAIN_DIR / f"train_{train_ts}.csv"
sample_all.to_csv(train_csv, index=False)

# Also keep a stable "latest" pointer you can reference in code
latest_csv = TRAIN_DIR / "train_latest.csv"
sample_all.to_csv(latest_csv, index=False)

print(f"✅ Saved training set (500 rows): {train_csv}")
print(f"🔁 Also updated: {latest_csv}")

# (Optional) Save per-class training files for class-specific experiments
PER_CLASS_DIR = TRAIN_DIR / f"per_class_{train_ts}"
PER_CLASS_DIR.mkdir(parents=True, exist_ok=True)

for cls in sample_all["label"].unique():
    out_cls = PER_CLASS_DIR / f"{cls}_train_{train_ts}.csv"
    sample_all[sample_all["label"] == cls].to_csv(out_cls, index=False)
    print(f"• Saved {cls} subset to: {out_cls}")

# (Optional) Keep a cumulative union of everything ever sampled (good for audit/repro)
CUMULATIVE_CSV = TRAIN_DIR / "all_selected_so_far.csv"
if CUMULATIVE_CSV.exists():
    prev = pd.read_csv(CUMULATIVE_CSV, low_memory=False)
    # Use unique_key to de-dup
    combined = pd.concat([prev, sample_all], ignore_index=True)
    combined = combined.drop_duplicates(subset=["unique_key"])
else:
    combined = sample_all

combined.to_csv(CUMULATIVE_CSV, index=False)
print(f"📚 Cumulative selected-so-far updated: {CUMULATIVE_CSV}")


✅ Saved training set (500 rows): training_sets/train_20251022_021946.csv
🔁 Also updated: training_sets/train_latest.csv
• Saved abortion subset to: training_sets/per_class_20251022_021946/abortion_train_20251022_021946.csv
• Saved harassment subset to: training_sets/per_class_20251022_021946/harassment_train_20251022_021946.csv
• Saved miscarriage subset to: training_sets/per_class_20251022_021946/miscarriage_train_20251022_021946.csv
📚 Cumulative selected-so-far updated: training_sets/all_selected_so_far.csv


In [9]:
sample_all.head(10)

,id,subreddit,title,selftext,created_utc,url,Tags,__dataset,__source_file,unique_key,score,num_comments,link_flair_text,over_18,strategy,label
0,1fyedxh,TwoXChromosomes,I took the abortion pill. I’m not okay.,"I’m 20 nearing 21, I’ve been in a committed re...",2024-10-07 18:10:45,https://www.reddit.com/r/TwoXChromosomes/comme...,NaN,abortion,abortion_data-updated - new_abortion_related_s...,faf2d7d073fbb5f24d13e2bdc7d09942d7a2ceb5b531ec...,NaN,NaN,NaN,NaN,NaN,abortion
1,1dxp81e,abortion,2nd abortion and I feel horrible,I feel like a scummy p.o.s. I had a medical a...,2024-07-07 19:53:40,https://www.reddit.com/r/abortion/comments/1dx...,NaN,abortion,abortion_data-updated - new_abortion_related_s...,44e7b6afa633404048de7009b2d2d0be6591631adcc1a6...,NaN,NaN,NaN,NaN,NaN,abortion
2,8cusgg,assault,am i allowed to feel guilty? does this even co...,basically i went on a “walk” last night with a...,2018-04-17 7:35:20,https://www.reddit.com/r/sexualassault/comment...,NaN,harassment,sexual-harrassment-data-updated - RelevantByTi...,d31fc0b3e2e27a40750058ca685fe3e8f09e445c08bf46...,NaN,NaN,NaN,NaN,NaN,harassment
3,1dzgb7t,abortion,Crying a few hours after abortion but I’m not ...,I was excited to have it done and i am so reli...,2024-07-09 23:01:40,https://www.reddit.com/r/abortion/comments/1dz...,NaN,abortion,abortion_data-updated - new_abortion_related_s...,7ea913805817fd88587697f49e392defe78b6c073558e4...,NaN,NaN,NaN,NaN,NaN,abortion
4,99yi99,assault,My Good Friend Assaulted Me,Writing this now is making me feel nauseous an...,2018-08-24 15:45:23,https://www.reddit.com/r/sexualassault/comment...,NaN,harassment,sexual-harrassment-data-updated - RelevantByTi...,4290adc718c6357211064015bf8c4421ed2d0d8b54b7e7...,NaN,NaN,NaN,NaN,NaN,harassment
5,asq5de,metoo,Getting This off my chest,Several people in my family know of my inciden...,2019-02-20 16:14:59,https://www.reddit.com/r/meToo/comments/asq5de...,NaN,harassment,sexual-harrassment-data-updated - RelevantByTi...,f322af71671318f44e06f521e113a031824579edf8427b...,NaN,NaN,NaN,NaN,NaN,harassment
6,fhls85,confession,I cut my insulin cord and could’ve been hospit...,"So back in 5th grade, I was really sad and I h...",2020-03-12 19:15:38,https://www.reddit.com/r/confession/comments/f...,NaN,abortion,abortion_data-updated - new_abortion_related_s...,bc69ffe70e26a227eaa4db3e577307cd1b23b5824fdff2...,NaN,NaN,NaN,NaN,NaN,abortion
7,1o6elw3,miscarriage,Missed miscarriage 11weeks,This was my first pregnancy and it was truly b...,2025-10-14T12:50:00,https://www.reddit.com/r/Miscarriage/comments/...,NaN,miscarriage,miscarriage_related_posts.csv,0f5f993db5b32d4e0c5f08c19c5e09e85cd04209d37785...,2.0,0.0,experience: first MC,False,new,miscarriage
8,ietc3s,abortion,Went to the ER,Two weeks ago I did a Medical Abortion at PP. ...,2020-08-23 0:01:14,https://www.reddit.com/r/abortion/comments/iet...,NaN,abortion,abortion_data-updated - new_abortion_related_s...,222e7ea69aaf5a815fafe11f9eee329bfd75e7a0ac5169...,NaN,NaN,NaN,NaN,NaN,abortion
9,1o6anuq,miscarriage,HCG numbers not doubling,I had a D&C in July at 9 weeks (MMC). I got a ...,2025-10-14T09:19:02,https://www.reddit.com/r/Miscarriage/comments/...,NaN,miscarriage,miscarriage_related_posts.csv,8245d4f0c679869bdc09846f659e03949f2b39d17bd390...,3.0,3.0,experience: D&C,False,new,miscarriage
